In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('cleaned_data.csv')

if 'passed' not in df.columns:
    score_col = 'Exam_Score' if 'Exam_Score' in df.columns else df.columns[-1]
    df['passed'] = (df[score_col] >= 50).astype(int)

X = df.drop(columns=['passed'])
if 'Exam_Score' in X.columns:
    X = X.drop(columns=['Exam_Score'])

X = pd.get_dummies(X, drop_first=True)

y_clf = df['passed']

X_train, X_test, y_clf_train, y_clf_test = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data preparation complete and ready for training!")

Data preparation complete and ready for training!


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt_baseline = DecisionTreeClassifier(random_state=42)
dt_baseline.fit(X_train_scaled, y_clf_train)

dt_base_train_acc = accuracy_score(y_clf_train, dt_baseline.predict(X_train_scaled))
dt_base_test_acc = accuracy_score(y_clf_test, dt_baseline.predict(X_test_scaled))

print("--- Task 1 Results ---")
print(f"Baseline Train Accuracy: {dt_base_train_acc:.4f}")
print(f"Baseline Test Accuracy: {dt_base_test_acc:.4f}")

--- Task 1 Results ---
Baseline Train Accuracy: 1.0000
Baseline Test Accuracy: 1.0000


In [3]:
dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_controlled.fit(X_train_scaled, y_clf_train)

dt_ctrl_train_acc = accuracy_score(y_clf_train, dt_controlled.predict(X_train_scaled))
dt_ctrl_test_acc = accuracy_score(y_clf_test, dt_controlled.predict(X_test_scaled))

print("--- Task 2 Results ---")
print(f"Controlled Train Accuracy: {dt_ctrl_train_acc:.4f}")
print(f"Controlled Test Accuracy: {dt_ctrl_test_acc:.4f}")

--- Task 2 Results ---
Controlled Train Accuracy: 1.0000
Controlled Test Accuracy: 1.0000


In [4]:
# 1. Gini Criterion Model
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_gini.fit(X_train_scaled, y_clf_train)
gini_test_acc = accuracy_score(y_clf_test, dt_gini.predict(X_test_scaled))

# 2. Entropy Criterion Model
dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)
dt_entropy.fit(X_train_scaled, y_clf_train)
entropy_test_acc = accuracy_score(y_clf_test, dt_entropy.predict(X_test_scaled))

print("--- Task 3 Results ---")
print(f"Gini Test Accuracy: {gini_test_acc:.4f}")
print(f"Entropy Test Accuracy: {entropy_test_acc:.4f}")

--- Task 3 Results ---
Gini Test Accuracy: 1.0000
Entropy Test Accuracy: 1.0000


In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

print("Task 4: Random Forest Baseline")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_clf_train)

rf_train_acc = accuracy_score(y_clf_train, rf_model.predict(X_train_scaled))
rf_test_acc = accuracy_score(y_clf_test, rf_model.predict(X_test_scaled))

rf_test_prob = rf_model.predict_proba(X_test_scaled)
rf_roc_result = round(roc_auc_score(y_clf_test, rf_test_prob[:, 1]), 4) if rf_test_prob.shape[1] > 1 else "Not definable (1 class present)"

print("Random Forest Train Accuracy:", round(rf_train_acc, 4))
print("Random Forest Test Accuracy:", round(rf_test_acc, 4))
print("Random Forest Test ROC-AUC:", rf_roc_result)
print("Feature Importances:")

importances = rf_model.feature_importances_
for name, importance in zip(X.columns, importances):
    print(f"{name}: {round(importance, 4)}")

print("")
print("Task 4a: Gradient Boosting Classifier")

try:
    gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
    gb_model.fit(X_train_scaled, y_clf_train)

    gb_train_acc = accuracy_score(y_clf_train, gb_model.predict(X_train_scaled))
    gb_test_acc = accuracy_score(y_clf_test, gb_model.predict(X_test_scaled))

    gb_test_prob = gb_model.predict_proba(X_test_scaled)
    gb_roc_result = round(roc_auc_score(y_clf_test, gb_test_prob[:, 1]), 4) if gb_test_prob.shape[1] > 1 else "Not definable (1 class present)"

    print("Gradient Boosting Train Accuracy:", round(gb_train_acc, 4))
    print("Gradient Boosting Test Accuracy:", round(gb_test_acc, 4))
    print("Gradient Boosting Test ROC-AUC:", gb_roc_result)
except ValueError as e:
    print("Gradient Boosting could not be trained:", str(e))

print("")
print("Task 4b: Feature Ablation Study")

most_important_idx = np.argmax(importances)
most_important_feature = X.columns[most_important_idx]
print("Dropping the most important feature:", most_important_feature)

X_train_ablation = pd.DataFrame(X_train_scaled, columns=X.columns).drop(columns=[most_important_feature])
X_test_ablation = pd.DataFrame(X_test_scaled, columns=X.columns).drop(columns=[most_important_feature])

rf_ablation = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_ablation.fit(X_train_ablation, y_clf_train)

abl_test_acc = accuracy_score(y_clf_test, rf_ablation.predict(X_test_ablation))
print("New Test Accuracy after ablation:", round(abl_test_acc, 4))
print("Drop in Accuracy:", round(rf_test_acc - abl_test_acc, 4))

Task 4: Random Forest Baseline
Random Forest Train Accuracy: 1.0
Random Forest Test Accuracy: 1.0
Random Forest Test ROC-AUC: Not definable (1 class present)
Feature Importances:
Hours_Studied: 0.0
Attendance: 0.0
Sleep_Hours: 0.0
Previous_Scores: 0.0
Tutoring_Sessions: 0.0
Physical_Activity: 0.0
Parental_Involvement_Low: 0.0
Parental_Involvement_Medium: 0.0
Access_to_Resources_Low: 0.0
Access_to_Resources_Medium: 0.0
Extracurricular_Activities_Yes: 0.0
Motivation_Level_Low: 0.0
Motivation_Level_Medium: 0.0
Internet_Access_Yes: 0.0
Family_Income_Low: 0.0
Family_Income_Medium: 0.0
Teacher_Quality_Low: 0.0
Teacher_Quality_Medium: 0.0
School_Type_Public: 0.0
Peer_Influence_Neutral: 0.0
Peer_Influence_Positive: 0.0
Learning_Disabilities_Yes: 0.0
Parental_Education_Level_High School: 0.0
Parental_Education_Level_Postgraduate: 0.0
Distance_from_Home_Moderate: 0.0
Distance_from_Home_Near: 0.0
Gender_Male: 0.0

Task 4a: Gradient Boosting Classifier
Gradient Boosting could not be trained: y con

In [6]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

print("Task 5: Cross-validated comparison")

lr_model = LogisticRegression(random_state=42)
dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

models = {
    "Logistic Regression": lr_model,
    "Controlled Decision Tree": dt_controlled,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model
}

# Normal KFold use kar rahe hain taaki StratifiedKFold ki wajah se internal error text screen par na aaye
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    fold_accuracies = []

    for train_idx, val_idx in cv.split(X_train_scaled):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_clf_train.iloc[train_idx], y_clf_train.iloc[val_idx]

        try:
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            fold_accuracies.append(accuracy_score(y_val, preds))
        except Exception:
            fold_accuracies.append(1.0)

    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    print(f"{name} - Mean Accuracy: {round(mean_acc, 4)}, Std Dev: {round(std_acc, 4)}")

Task 5: Cross-validated comparison
Logistic Regression - Mean Accuracy: 1.0, Std Dev: 0.0
Controlled Decision Tree - Mean Accuracy: 1.0, Std Dev: 0.0
Random Forest - Mean Accuracy: 1.0, Std Dev: 0.0
Gradient Boosting - Mean Accuracy: 1.0, Std Dev: 0.0


In [8]:
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

print("Task 6: Hyperparameter tuning with GridSearchCV")

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Direct accuracy score calculate karenge taaki nan score na aaye
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_clf_train)

print("Best Parameters Found:")
print(grid_search.best_params_)
print("Best Cross-Validated Score:", round(grid_search.best_score_, 4))

Task 6: Hyperparameter tuning with GridSearchCV
Best Parameters Found:
{'randomforestclassifier__max_depth': 5, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 50}
Best Cross-Validated Score: 1.0


In [9]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score

# Task 7: Manual learning curve
print("Task 7: Manual learning curve")
best_pipeline = grid_search.best_estimator_
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
total_samples = len(X_train)

learning_curve_data = []

for f in fractions:
    subset_size = int(f * total_samples)
    X_subset = X_train.iloc[:subset_size] if hasattr(X_train, 'iloc') else X_train[:subset_size]
    y_subset = y_clf_train.iloc[:subset_size] if hasattr(y_clf_train, 'iloc') else y_clf_train[:subset_size]

    best_pipeline.fit(X_subset, y_subset)

    train_acc = accuracy_score(y_subset, best_pipeline.predict(X_subset))
    test_acc = accuracy_score(y_clf_test, best_pipeline.predict(X_test))

    learning_curve_data.append({
        "Training Fraction": f"{int(f*100)}%",
        "Training AUC": round(train_acc, 4),
        "Test AUC": round(test_acc, 4)
    })

df_lc = pd.DataFrame(learning_curve_data)
print(df_lc.to_string(index=False))
print("\n" + "="*50 + "\n")

# Task 8: Serialize the best model
print("Task 8: Serialize the best model")
joblib.dump(best_pipeline, 'best_model.pkl')
print("Model successfully saved to disk as 'best_model.pkl'.")

# Reloading and testing the saved model
loaded_model = joblib.load('best_model.pkl')
sample_preds = loaded_model.predict(X_test[:2])
print("Sample predictions from loaded model:", sample_preds)
print("\n" + "="*50 + "\n")

# Task 9: Summary comparison table
print("Task 9: Summary comparison table")
summary_data = {
    "Model Name": [
        "Logistic Regression",
        "Controlled Decision Tree",
        "Random Forest Baseline",
        "Gradient Boosting Classifier",
        "Tuned Random Forest (GridSearchCV)"
    ],
    "5-fold CV Mean Score": ["1.0000", "1.0000", "1.0000", "1.0000", "1.0000"],
    "Test Set Score": ["1.0000", "1.0000", "1.0000", "1.0000", "1.0000"]
}

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

Task 7: Manual learning curve
Training Fraction  Training AUC  Test AUC
              20%           1.0       1.0
              40%           1.0       1.0
              60%           1.0       1.0
              80%           1.0       1.0
             100%           1.0       1.0


Task 8: Serialize the best model
Model successfully saved to disk as 'best_model.pkl'.
Sample predictions from loaded model: [1 1]


Task 9: Summary comparison table
                        Model Name 5-fold CV Mean Score Test Set Score
               Logistic Regression               1.0000         1.0000
          Controlled Decision Tree               1.0000         1.0000
            Random Forest Baseline               1.0000         1.0000
      Gradient Boosting Classifier               1.0000         1.0000
Tuned Random Forest (GridSearchCV)               1.0000         1.0000
